# DuckDB engine: large dataset check

Run the DuckDB check on a typed Parquet file and inspect the overview table and the full per column report.

In [1]:
import os, sys

# Make the project root importable (engines, utils) no matter where the kernel starts.
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "engines")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import warnings
warnings.filterwarnings("ignore")

import duckdb

from engines.core import inspect_data
from utils.notebook_utils import show_struct

DATA_PATH = os.path.join(_root, "example_data", "example-sales_data.parquet")

## The dataset
The sales example data, stored as typed Parquet.

In [7]:
n_rows = duckdb.execute(f"SELECT COUNT(*) FROM read_parquet('{DATA_PATH}')").fetchone()[0]
columns = [row[0] for row in duckdb.execute(f"DESCRIBE SELECT * FROM read_parquet('{DATA_PATH}')").fetchall()]

print(f"Rows: {n_rows:,}")
print(f"Columns: {len(columns)}")
print(f"Column names: {columns}")

Rows: 2,100
Columns: 11
Column names: ['Order_ID', 'Date', 'Customer_Name', 'Email', 'Age', 'City', 'Product', 'Category', 'Price', 'Quantity', 'Payment_Method']


## Run the check
One call runs the check and returns an overview table and a detailed report.

In [4]:
summary_table, report = inspect_data(DATA_PATH, engine="duckdb")

## Overview table
One row per column: missing values, outliers and uniqueness at a glance.

In [5]:
summary_table

,column,dtype,n_values,n_missing,pct_missing,n_present,n_wrong_dtype,pct_wrong_dtype,n_negative,n_outlier_low,n_outlier_high,n_unique,pct_unique
0,Order_ID,INTEGER,2100,0,0.00,2100,None,None,0.0,0.0,0.0,2000.0,95.24
1,Date,DATE,2100,0,0.00,2100,None,None,NaN,NaN,NaN,NaN,NaN
2,Customer_Name,VARCHAR,2100,36,1.71,2064,None,None,NaN,NaN,NaN,920.0,43.81
3,Email,VARCHAR,2100,49,2.33,2051,None,None,NaN,NaN,NaN,812.0,38.67
4,Age,INTEGER,2100,136,6.48,1964,None,None,19.0,0.0,30.0,NaN,NaN
5,City,VARCHAR,2100,36,1.71,2064,None,None,NaN,NaN,NaN,63.0,3.00
6,Product,VARCHAR,2100,0,0.00,2100,None,None,NaN,NaN,NaN,10.0,0.48
7,Category,VARCHAR,2100,40,1.90,2060,None,None,NaN,NaN,NaN,10.0,0.48
8,Price,DOUBLE,2100,96,4.57,2004,None,None,43.0,34.0,385.0,NaN,NaN
9,Quantity,INTEGER,2100,44,2.10,2056,None,None,60.0,11.0,21.0,NaN,NaN


## Full report per column
Expand any column to see its detailed diagnostics.

In [6]:
show_struct(report)